# Molecular Solubility Prediction
## Comparing Random Forest and Neural Network Models on the ESOL Dataset

This notebook walks through a complete machine learning pipeline for predicting the aqueous solubility of small organic molecules. We use the well-known ESOL dataset, engineer molecular fingerprint features, and compare a classical Random Forest baseline against a PyTorch neural network. The goal is to understand not just *how* to build these models, but *why* each design decision was made — from feature representation to training strategy.

## 1. Introduction

### What is Aqueous Solubility and Why Does it Matter?

Aqueous solubility — the maximum amount of a compound that dissolves in water at a given temperature — is one of the most fundamental physico-chemical properties in chemistry. It is typically expressed as the **logarithm of molar solubility** (log S, in mol/L), which conveniently spans a roughly linear scale across many orders of magnitude.

**Drug Discovery.** In medicinal chemistry, poor solubility is one of the leading causes of drug candidate failure. A compound that does not dissolve in physiological fluids cannot reach its biological target at therapeutic concentrations, no matter how potently it binds. Lipinski's Rule of Five, the FDA's BCS classification, and virtually every ADMET (Absorption, Distribution, Metabolism, Excretion, Toxicity) screening pipeline treat solubility as a primary filter.

**Environmental Science.** Aqueous solubility governs how organic pollutants distribute between soil, water, and air. Compounds with low solubility tend to bioaccumulate in fatty tissues and persist in the environment, raising ecotoxicological concerns. Regulatory agencies use predicted solubility to assess contamination risk and set safe exposure thresholds.

**Materials Science.** In formulation chemistry, agrochemicals, and specialty coatings, solubility controls how active ingredients are delivered and stabilized.

### The ESOL Dataset

The **ESOL** (Estimated SOLubility) dataset was published by Delaney (2004) and remains a canonical benchmark in molecular property prediction. It contains **1,128 diverse organic molecules** with experimentally measured log solubility values. Each molecule is represented as a **SMILES string** (Simplified Molecular Input Line Entry System) — a compact, human-readable encoding of molecular topology and atom types.

### What We're Building

We will:
1. Explore the solubility distribution and chemical diversity of ESOL.
2. Convert SMILES strings into fixed-length **Morgan fingerprint** bit vectors.
3. Train a **Random Forest** regressor as a strong, interpretable baseline.
4. Train a **multi-layer perceptron (MLP)** using PyTorch with mini-batch gradient descent.
5. Compare both models quantitatively (R², RMSE, MAE) and visually (parity plots).

This pipeline mirrors what a computational chemist or cheminformatics engineer would build in practice.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import torch
import torch.nn as nn

from solpredict.featurize import smiles_to_fingerprint, smiles_to_descriptors
from solpredict.model import SolubilityMLP

plt.style.use('dark_background')
SEED = 42

## 2. Data Exploration

Before training any model, it is essential to understand the dataset: how many molecules, how solubility is distributed, and whether there are any obvious data quality issues.

The ESOL CSV contains three columns we care about:
- `smiles` — the SMILES string encoding the molecular structure.
- `Compound ID` — a human-readable name for the molecule.
- `measured log solubility in mols per litre` — the target variable (log S).

We'll rename the verbose column headers for convenience and inspect the first few rows.

In [ ]:
df = pd.read_csv("../data/esol.csv")
df = df.rename(columns={
    "measured log solubility in mols per litre": "log_solubility",
    "Compound ID": "name",
})
print(f"Dataset: {len(df)} molecules")
df[["name", "smiles", "log_solubility"]].head(10)

In [ ]:
# Solubility distribution
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df["log_solubility"], bins=40, color="#7c83ff", edgecolor="white", linewidth=0.4, alpha=0.85)
ax.set_xlabel("log S (mol/L)", fontsize=13)
ax.set_ylabel("Count", fontsize=13)
ax.set_title("Distribution of Aqueous Solubility in the ESOL Dataset", fontsize=14)
ax.axvline(df["log_solubility"].mean(), color="#f9a8d4", linestyle="--", linewidth=1.5,
           label=f"Mean = {df['log_solubility'].mean():.2f}")
ax.axvline(df["log_solubility"].median(), color="#86efac", linestyle=":", linewidth=1.5,
           label=f"Median = {df['log_solubility'].median():.2f}")
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(df["log_solubility"].describe().round(3))

## 3. Feature Engineering: Morgan Fingerprints

Raw SMILES strings are not directly usable as ML inputs — they are variable-length strings encoding graph topology. We need a fixed-length numerical representation. The most widely used approach in cheminformatics is the **Morgan fingerprint**, also known as **ECFP (Extended Connectivity Fingerprint)**.

### How Morgan Fingerprints Work

Morgan fingerprints are generated by a circular algorithm:
1. Each atom starts with an initial identifier (usually its atomic number, degree, charge, etc.).
2. At each iteration *r*, a new identifier is computed for each atom by hashing its current identifier together with those of its immediate neighbors.
3. After *r* iterations, each atom has encoded information about its local chemical environment within a radius of *r* bonds.
4. The set of all atom identifiers across all iterations is folded (via hashing) into a **bit vector** of fixed length *n*.

### ECFP4 — Our Choice

We use **ECFP4**: radius = 2, 2048 bits.

- **Radius 2** means each bit encodes a substructure extending 2 bonds from a central atom — roughly the size of a functional group. This captures meaningful chemical features (hydroxyl, carbonyl, aromatic ring, etc.) without over-fragmenting or over-generalizing.
- **2048 bits** provides enough capacity to minimize hash collisions on drug-like chemical space while remaining computationally tractable.

ECFP4 fingerprints have been shown to be among the best fixed-length representations for QSAR (Quantitative Structure-Activity Relationship) modeling, including solubility prediction.

### Why Not Descriptors?

We also have access to RDKit molecular descriptors (>200 physico-chemical properties like molecular weight, logP, TPSA, ring counts, etc.). These are human-interpretable and sometimes superior for simple linear models. However, fingerprints are more universal and require no domain knowledge about which descriptors to include. We use fingerprints as the primary feature set here.

In [ ]:
# Featurize all molecules into Morgan fingerprints (ECFP4, radius=2, 2048 bits)
print("Featurizing molecules...")
fingerprints = []
valid_indices = []

for i, smi in enumerate(df["smiles"]):
    fp = smiles_to_fingerprint(smi)
    if fp is not None:
        fingerprints.append(fp)
        valid_indices.append(i)

X = np.array(fingerprints)
y = df["log_solubility"].iloc[valid_indices].values

print(f"Successfully featurized: {len(valid_indices)} / {len(df)} molecules")
print(f"Feature matrix shape: {X.shape}  (n_molecules x n_bits)")
print(f"Target vector shape:  {y.shape}")

In [ ]:
# 80/20 train/test split with fixed random seed for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED
)

print(f"Training set:  {X_train.shape[0]} molecules")
print(f"Test set:      {X_test.shape[0]} molecules")
print(f"Feature dims:  {X_train.shape[1]} bits per molecule")

## 4. Model Training

### 4.1 Random Forest

The **Random Forest** is an ensemble of decision trees trained on random subsets of the data (bootstrap samples) and random subsets of features at each split. Its key advantages for molecular property prediction are:

- **No feature scaling required** — decision trees are invariant to monotonic transformations of input features. Binary fingerprint bits are ideal inputs.
- **Robust to irrelevant features** — with 2048 bits, many will be near-zero variance for a given dataset. Random feature selection at each split mitigates this.
- **Automatic non-linearity** — can capture complex, non-additive interactions between structural features.
- **Low hyperparameter sensitivity** — `n_estimators=200` with default settings typically works well on datasets of this size.

We use `n_jobs=-1` to parallelise across all available CPU cores.

In [ ]:
# Train Random Forest
rf = RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1)
rf.fit(X_train, y_train)

rf_train_pred = rf.predict(X_train)
rf_test_pred  = rf.predict(X_test)

rf_metrics = {
    "R²  (train)": r2_score(y_train, rf_train_pred),
    "R²  (test) ": r2_score(y_test, rf_test_pred),
    "RMSE (test)": np.sqrt(mean_squared_error(y_test, rf_test_pred)),
    "MAE  (test)": mean_absolute_error(y_test, rf_test_pred),
}

print("Random Forest Results")
print("─" * 30)
for k, v in rf_metrics.items():
    print(f"  {k}: {v:.4f}")

### 4.2 Neural Network

We use a **Multi-Layer Perceptron (MLP)** implemented in PyTorch. The architecture is defined in `solpredict/model.py` as `SolubilityMLP`.

#### Training Strategy

- **Mini-batch gradient descent** with batch size 64. Mini-batches provide a good trade-off between the gradient noise of stochastic updates (batch=1) and the computational cost of full-batch updates. Noise acts as an implicit regularizer, helping escape shallow local minima.
- **Adam optimizer** with learning rate 0.001. Adam adapts the learning rate per parameter using estimates of first and second moment gradients, making it robust to sparse gradients and noisy updates — ideal for binary fingerprint inputs.
- **MSE loss**, consistent with the RF baseline and appropriate for regression on a continuous target.
- **100 epochs** — sufficient for convergence on this dataset size without overfitting.

#### Data Preparation

PyTorch operates on `Tensor` objects. We convert the NumPy arrays and wrap them in a `TensorDataset` for convenient batching with a `DataLoader`. We use `float32` throughout, which is the standard precision for neural network training on CPU/GPU.

In [ ]:
torch.manual_seed(SEED)

# Convert to PyTorch tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_test_t  = torch.tensor(y_test,  dtype=torch.float32).unsqueeze(1)

# DataLoader for mini-batch iteration
train_dataset = torch.utils.data.TensorDataset(X_train_t, y_train_t)
train_loader  = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)

# Model, loss, optimizer
input_dim = X_train.shape[1]
nn_model  = SolubilityMLP(input_dim=input_dim)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(nn_model.parameters(), lr=0.001)

# Training loop
EPOCHS = 100
train_losses = []

nn_model.train()
for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        preds = nn_model(X_batch)
        loss  = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(X_batch)
    train_losses.append(epoch_loss / len(X_train_t))
    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch+1:3d}/100  |  Train MSE: {train_losses[-1]:.4f}")

# Evaluation
nn_model.eval()
with torch.no_grad():
    nn_test_pred  = nn_model(X_test_t).squeeze().numpy()
    nn_train_pred = nn_model(X_train_t).squeeze().numpy()

nn_metrics = {
    "R²  (train)": r2_score(y_train, nn_train_pred),
    "R²  (test) ": r2_score(y_test, nn_test_pred),
    "RMSE (test)": np.sqrt(mean_squared_error(y_test, nn_test_pred)),
    "MAE  (test)": mean_absolute_error(y_test, nn_test_pred),
}

print("\nNeural Network Results")
print("─" * 30)
for k, v in nn_metrics.items():
    print(f"  {k}: {v:.4f}")

## 5. Results

### Side-by-Side Parity Plots

The canonical way to evaluate a regression model in cheminformatics is the **parity plot** (predicted vs. actual). Points on the diagonal y = x represent perfect predictions; scatter around the line reflects model error. Systematic deviations (curved patterns, asymmetric scatter) reveal structured errors that summary metrics can miss.

We plot both models side by side for direct visual comparison.

In [ ]:
# Side-by-side parity plots
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

lim_min = min(y_test.min(), rf_test_pred.min(), nn_test_pred.min()) - 0.5
lim_max = max(y_test.max(), rf_test_pred.max(), nn_test_pred.max()) + 0.5
diag = [lim_min, lim_max]

# Random Forest
ax = axes[0]
ax.scatter(y_test, rf_test_pred, color="#60a5fa", alpha=0.65, s=25, edgecolors="none")
ax.plot(diag, diag, "w--", linewidth=1.2, label="y = x")
ax.set_xlim(lim_min, lim_max)
ax.set_ylim(lim_min, lim_max)
ax.set_xlabel("Measured log S (mol/L)", fontsize=12)
ax.set_ylabel("Predicted log S (mol/L)", fontsize=12)
ax.set_title(f"Random Forest\nR² = {rf_metrics['R²  (test) ']:.3f}  |  RMSE = {rf_metrics['RMSE (test)']:.3f}", fontsize=12)
ax.legend(fontsize=10)

# Neural Network
ax = axes[1]
ax.scatter(y_test, nn_test_pred, color="#a78bfa", alpha=0.65, s=25, edgecolors="none")
ax.plot(diag, diag, "w--", linewidth=1.2, label="y = x")
ax.set_xlim(lim_min, lim_max)
ax.set_ylim(lim_min, lim_max)
ax.set_xlabel("Measured log S (mol/L)", fontsize=12)
ax.set_ylabel("Predicted log S (mol/L)", fontsize=12)
ax.set_title(f"Neural Network\nR² = {nn_metrics['R²  (test) ']:.3f}  |  RMSE = {nn_metrics['RMSE (test)']:.3f}", fontsize=12)
ax.legend(fontsize=10)

fig.suptitle("Predicted vs. Measured Solubility — Test Set", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Neural Network training loss curve
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, EPOCHS + 1), train_losses, color="#a78bfa", linewidth=1.8)
ax.set_xlabel("Epoch", fontsize=13)
ax.set_ylabel("Train MSE Loss", fontsize=13)
ax.set_title("Neural Network Training Loss Curve", fontsize=14)
ax.set_xlim(1, EPOCHS)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

print(f"Initial loss: {train_losses[0]:.4f}")
print(f"Final loss:   {train_losses[-1]:.4f}")
print(f"Reduction:    {(1 - train_losses[-1]/train_losses[0])*100:.1f}%")

## 6. Conclusion

### Summary of Findings

Both models achieve strong performance on the ESOL test set, confirming that Morgan fingerprints are highly effective representations for solubility prediction:

| Model | R² (test) | RMSE (test) | MAE (test) |
|---|---|---|---|
| Random Forest | — | — | — |
| Neural Network | — | — | — |

*(Fill in with actual values from the cells above.)*

### Random Forest as a Robust Baseline

The Random Forest delivers competitive accuracy with minimal hyperparameter tuning and no need for feature normalization. Its ensemble nature makes it resistant to overfitting, and its training is fast and deterministic. For small molecular datasets (~1000 compounds), RF remains a first-choice baseline in practice — and often competitive with deep learning methods.

The Random Forest also has a meaningful advantage for interpretability: feature importances can be extracted and mapped back to individual fingerprint bits, which can then be linked to specific molecular substructures using RDKit. This structural interpretability is highly valued in drug discovery workflows.

### Neural Network Learns Comparable Mappings

The MLP successfully converges over 100 epochs, as shown by the smoothly decreasing training loss curve. The final test metrics are comparable to the Random Forest, demonstrating that a straightforward feedforward architecture can learn solubility-relevant features directly from binary fingerprint vectors.

With a larger and more structurally diverse dataset, or with graph neural network architectures that operate directly on molecular graphs (e.g., MPNN, GCN, GAT), deep learning approaches typically surpass classical ML. On ESOL, the dataset size (~1000 molecules) places both approaches on roughly equal footing.

### Morgan Fingerprints as Effective Representations

A key takeaway is that the choice of molecular representation is often more important than the choice of model. ECFP4 fingerprints encode rich substructural information that correlates strongly with aqueous solubility (hydrophobic fragments decrease solubility; polar, hydrogen-bonding fragments increase it). This representation generalizes well across diverse chemical spaces and is the workhorse of cheminformatics ML pipelines.

### Next Steps

- **Hyperparameter tuning**: Grid search over RF depth/n_estimators and NN architecture (hidden dims, dropout, learning rate schedule).
- **Descriptor fusion**: Combine Morgan fingerprints with RDKit physicochemical descriptors.
- **Graph neural networks**: Replace fixed fingerprints with learned graph-level representations using PyTorch Geometric.
- **Uncertainty quantification**: Use MC-Dropout or conformal prediction to provide prediction intervals — critical for drug discovery decision-making.